In [4]:
import pandas as pd
import json
import numpy as np
from pathlib import Path

In [7]:
from pathlib import Path
import json
import numpy as np
import os

# 1. CORREÇÃO DO CAMINHO (Notebooks não têm __file__)
# Usamos o diretório de trabalho atual.
# Geralmente o notebook roda na raiz ou em /src. Verifique com print(current_dir)
current_dir = Path.cwd()

# Tenta encontrar a raiz do projeto procurando pela pasta 'data'
# Se não encontrar, assume que estamos na raiz ou ajusta manualmente
if (current_dir / "data").exists():
    project_root = current_dir
elif (current_dir.parent / "data").exists():
    project_root = current_dir.parent
else:
    # Fallback: defina manualmente se necessário
    # project_root = Path("D:/UFC/.../THP7244-final-project")
    project_root = current_dir.parent.parent # Ajuste conforme profundidade da pasta

print(f"Raiz do projeto detectada: {project_root}")

# 2. CORREÇÃO DE LÓGICA: Carregar o JSON, não o DSS
# O arquivo que tem a estrutura data['lines'] é o JSON que você gerou antes
json_file_path = "IEEE13Nodeckt.json" 

# Se o json não estiver na mesma pasta do notebook, construa o caminho completo:
# json_file_path = project_root / "rede_eletrica.json"

if not os.path.exists(json_file_path):
    print(f"ERRO: Arquivo {json_file_path} não encontrado. Gere-o primeiro com a classe OpenDSS2LinDist3Flow.")
else:
    # Carregar JSON (agora sim funcionará)
    with open(json_file_path, 'r') as f:
        data = json.load(f)

    print("JSON carregado com sucesso. Processando dados...")

    # Converter listas de volta para numpy arrays
    lines_processed = []
    for line in data['lines']:
        line['r_matrix'] = np.array(line['r_matrix'])
        line['x_matrix'] = np.array(line['x_matrix'])
        lines_processed.append(line)

    # Configurar Cargas como matriz (3, N)
    nodes = data['nodes']
    node_map = {n: i for i, n in enumerate(nodes)}
    n_nodes = len(nodes)

    load_p = np.zeros((3, n_nodes))
    load_q = np.zeros((3, n_nodes))

    # Converter kW para p.u. (precisa dividir pelo S_base do JSON)
    s_base_mva = data['general']['s_base_mva']
    s_base_kw = s_base_mva * 1000

    for load in data['loads']:
        # Verifica se a barra da carga existe no mapeamento (segurança)
        if load['bus'] in node_map:
            idx = node_map[load['bus']]
            # Divide por s_base_kw para ter em p.u.
            load_p[:, idx] = np.array(load['p_load']) / s_base_kw
            load_q[:, idx] = np.array(load['q_load']) / s_base_kw
            
    print("Dados processados com sucesso. Matrizes prontas.")

Raiz do projeto detectada: d:\UFC\2-MESTRADO\1-semestre\ESTUDOS ESPECIAIS EM ENGENHARIA ELETRICA II (THP7244)\lucas\trabalho-final\projeto\THP7244-final-project
ERRO: Arquivo IEEE13Nodeckt.json não encontrado. Gere-o primeiro com a classe OpenDSS2LinDist3Flow.


In [ ]:
input = {
    'load-load1':
    {'file': '../data/14885589/0-id_114.csv',
     'type': 'load',
     'column': 'load power (kW)',
     'time-stamp': 60*5
     },

    'pv-pv1':
    {'file': '../data/14885589/0-id_114.csv',
     'type': 'pv',
     'column': 'solar power (kW)',
     'time-stamp': 60*5
    },

    'bess-bess1':
    {'file': '../data/14885589/0-id_114.csv',
     'type': 'bess',
     'column': 'battery power (kW)',
     'time-stamp': 60*5
    },
}